In [2]:
import pandas as pd
import glob

# data/raw 폴더 안의 모든 statcast csv 파일 찾기
files = glob.glob("../data/raw/statcast_*.csv")
print(f"파일 개수: {len(files)}")
print(files[:5])  # 처음 5개만 확인

파일 개수: 44
['../data/raw\\statcast_2021-03.csv', '../data/raw\\statcast_2021-04.csv', '../data/raw\\statcast_2021-05.csv', '../data/raw\\statcast_2021-06.csv', '../data/raw\\statcast_2021-07.csv']


In [3]:
# 파일 하나 불러와서 구조 확인
df_test = pd.read_csv(files[0])
print(df_test.shape)
print(df_test['pitch_type'].value_counts())

(42561, 119)
pitch_type
FF    8319
SL    4252
SI    3724
CH    2797
CU    1908
FC    1492
KC     533
FS     409
ST     190
SV     122
CS      11
Name: count, dtype: int64


In [4]:
# 모든 파일 합치기
df_list = [pd.read_csv(f) for f in files]
df_all = pd.concat(df_list, ignore_index=True)
print(df_all.shape)

# 패스트볼 3종만 필터링
target_pitches = ['FF', 'SI', 'FC']
df_fb = df_all[df_all['pitch_type'].isin(target_pitches)].copy()
print(df_fb.shape)
print(df_fb['pitch_type'].value_counts())
print(df_fb['game_year'].value_counts().sort_index())

(3846144, 119)
(2100398, 119)
pitch_type
FF    1233851
SI     579812
FC     286735
Name: count, dtype: int64
game_year
2021    431197
2022    421148
2023    419124
2024    415230
2025    413699
Name: count, dtype: int64


In [5]:
# 클러스터링에 쓸 피처 선택
features = ['release_speed', 'pfx_x', 'pfx_z', 'release_spin_rate',
            'spin_axis', 'release_pos_x', 'release_pos_z', 'release_extension']

# 결측치 확인
print(df_fb[features].isna().sum())

# 결측치 있는 행 제거
df_clean = df_fb.dropna(subset=features).copy()
print(f"\n결측 제거 전: {len(df_fb)}, 제거 후: {len(df_clean)}")

release_speed          20
pfx_x                  98
pfx_z                  21
release_spin_rate    9406
spin_axis            9425
release_pos_x         164
release_pos_z         164
release_extension    3903
dtype: int64

결측 제거 전: 2100398, 제거 후: 2090319


In [6]:
from sklearn.preprocessing import StandardScaler

X = df_clean[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)
print(X_scaled[:3])  # 스케일링된 값 미리보기

(2090319, 8)
[[-0.60867503  1.10527486  0.71880854 -0.93655252 -0.94075713  1.57543496
   0.79287068  1.07710479]
 [-0.1696068   0.82666446  0.36323004 -1.35426022 -0.94075713  1.64556594
   0.79287068  1.07710479]
 [ 0.36354748 -1.30531081 -1.0368603  -0.27131434  1.36738163 -0.89533335
  -0.65284175  0.18108391]]


In [7]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
import numpy as np

# 계산 시간 절약을 위해 5만 개 랜덤 샘플링
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), size=50000, replace=False)
X_sample = X_scaled[sample_idx]

# k=2부터 6까지 돌려보면서 실루엣 계수 확인
results = {}
for k in range(2, 7):
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=1)
    labels = gmm.fit_predict(X_sample)
    score = silhouette_score(X_sample, labels)
    results[k] = score
    print(f"k={k}: silhouette={score:.4f}")

k=2: silhouette=0.2820
k=3: silhouette=0.1749
k=4: silhouette=0.1971
k=5: silhouette=0.1800
k=6: silhouette=0.1560


In [8]:
years = sorted(df_clean['game_year'].unique())
year_results = {}

for year in years:
    df_year = df_clean[df_clean['game_year'] == year]
    X_year = scaler.transform(df_year[features].values)
    
    # 연도별 표본 2만개 샘플링 (속도 위해)
    np.random.seed(42)
    idx = np.random.choice(len(X_year), size=min(20000, len(X_year)), replace=False)
    X_year_sample = X_year[idx]
    
    scores = {}
    for k in range(2, 7):
        gmm = GaussianMixture(n_components=k, random_state=42, n_init=1)
        labels = gmm.fit_predict(X_year_sample)
        score = silhouette_score(X_year_sample, labels)
        scores[k] = score
    
    year_results[year] = scores
    best_k = max(scores, key=scores.get)
    print(f"{year}년 - best k={best_k} (silhouette={scores[best_k]:.4f})")
    print(f"   전체: {scores}")

2021년 - best k=2 (silhouette=0.2898)
   전체: {2: 0.2897597373521725, 3: 0.18443664177295618, 4: 0.19221250350325617, 5: 0.1568109903570219, 6: 0.13470438021288786}
2022년 - best k=2 (silhouette=0.2777)
   전체: {2: 0.27772348005738495, 3: 0.17620498552804598, 4: 0.19942905471796554, 5: 0.18288890962036372, 6: 0.1617562292521273}
2023년 - best k=2 (silhouette=0.2757)
   전체: {2: 0.2756649413740002, 3: 0.25895562303837116, 4: 0.19466591465444089, 5: 0.1760933545414621, 6: 0.1658408236786636}
2024년 - best k=2 (silhouette=0.2827)
   전체: {2: 0.2826975743267094, 3: 0.16947359742726567, 4: 0.20000739366035583, 5: 0.18224722186244757, 6: 0.1719290651666151}
2025년 - best k=2 (silhouette=0.2959)
   전체: {2: 0.2958505761636865, 3: 0.1841569923736934, 4: 0.20512295353250462, 5: 0.17348450436392657, 6: 0.16395458650803932}


In [9]:
from sklearn.model_selection import KFold

cv_results = {}

for year in years:
    df_year = df_clean[df_clean['game_year'] == year]
    X_year = scaler.transform(df_year[features].values)
    
    np.random.seed(42)
    idx = np.random.choice(len(X_year), size=min(20000, len(X_year)), replace=False)
    X_year_sample = X_year[idx]
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []
    
    for train_idx, test_idx in kf.split(X_year_sample):
        X_train, X_test = X_year_sample[train_idx], X_year_sample[test_idx]
        
        gmm = GaussianMixture(n_components=2, random_state=42, n_init=1)
        gmm.fit(X_train)
        test_labels = gmm.predict(X_test)
        
        # 테스트 fold에 클러스터가 1개만 나오면 실루엣 계산 불가하니 예외처리
        if len(set(test_labels)) > 1:
            score = silhouette_score(X_test, test_labels)
            fold_scores.append(score)
    
    cv_results[year] = fold_scores
    print(f"{year}년 - CV 평균 실루엣: {np.mean(fold_scores):.4f} (fold별: {[f'{s:.3f}' for s in fold_scores]})")

2021년 - CV 평균 실루엣: 0.2896 (fold별: ['0.290', '0.291', '0.290', '0.290', '0.286'])
2022년 - CV 평균 실루엣: 0.2774 (fold별: ['0.276', '0.272', '0.286', '0.276', '0.277'])
2023년 - CV 평균 실루엣: 0.2757 (fold별: ['0.275', '0.274', '0.275', '0.278', '0.276'])
2024년 - CV 평균 실루엣: 0.2823 (fold별: ['0.285', '0.281', '0.281', '0.285', '0.279'])
2025년 - CV 평균 실루엣: 0.2961 (fold별: ['0.293', '0.294', '0.301', '0.304', '0.289'])


In [10]:
# k=2로 최종 클러스터링 (전체 샘플 기준)
gmm_final = GaussianMixture(n_components=2, random_state=42, n_init=1)
cluster_labels = gmm_final.fit_predict(X_sample)

# 샘플에 해당하는 원본 데이터 가져오기
df_sample = df_clean.iloc[sample_idx].copy()
df_sample['cluster'] = cluster_labels

# 클러스터별 구종 분포
print("클러스터별 구종 분포:")
print(pd.crosstab(df_sample['cluster'], df_sample['pitch_type']))

print("\n클러스터별 평균 특성:")
print(df_sample.groupby('cluster')[features].mean().round(2))

클러스터별 구종 분포:
pitch_type    FC     FF    SI
cluster                      
0            959  20665  9478
1           6019   8654  4225

클러스터별 평균 특성:
         release_speed  pfx_x  pfx_z  release_spin_rate  spin_axis  \
cluster                                                              
0                94.22  -0.80   1.12            2260.33     216.57   
1                91.62   0.61   0.95            2273.40     153.42   

         release_pos_x  release_pos_z  release_extension  
cluster                                                   
0                -1.85           5.76               6.45  
1                 1.02           5.87               6.37  
